In [ ]:
from pathlib import Path

import torch
import yaml
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video
from omegaconf import OmegaConf

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

In [ ]:
pipe = CogVideoXPipeline.from_pretrained(
    "/hf_models",
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
).to("cuda")
# pipe.enable_model_cpu_offload()
# pipe.vae.enable_tiling()

In [ ]:
config_path = Path("../configs/baseline_prompts.yaml")
cfg = OmegaConf.load(config_path)

output_dir = Path("../data/baseline")
output_dir.mkdir(parents=True, exist_ok=True)

gen = cfg.generation
prompts = cfg.prompts

print(f"Loaded {len(prompts)} baseline prompts from {config_path}")
for p in prompts:
    print(f"  [{p.motion_profile:16s}] {p.id}")

In [ ]:
manifest = []

with torch.inference_mode():
    for prompt_cfg in prompts:
        prompt_id = prompt_cfg.id
        out_path = output_dir / f"{prompt_id}.mp4"

        print(f"\n=== {prompt_id} ({prompt_cfg.motion_profile}) ===")
        result = pipe(
            prompt=prompt_cfg.text,
            negative_prompt=gen.negative_prompt,
            num_frames=gen.num_frames,
            num_inference_steps=gen.num_inference_steps,
            guidance_scale=gen.guidance_scale,
        )
        export_to_video(result.frames[0], str(out_path), fps=gen.fps)

        manifest.append(
            {
                "id": prompt_id,
                "category": prompt_cfg.category,
                "motion_profile": prompt_cfg.motion_profile,
                "prompt": prompt_cfg.text,
                "video_path": str(out_path),
            }
        )
        print(f"saved -> {out_path}")

manifest_path = output_dir / "manifest.yaml"
with manifest_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(manifest, f, allow_unicode=True, sort_keys=False)

print(f"\nWrote manifest -> {manifest_path}")

In [ ]:
# Quick smoke test for a single prompt (uncomment to run one video only)
# smoke = prompts[0]
# result = pipe(
#     prompt=smoke.text,
#     negative_prompt=gen.negative_prompt,
#     num_frames=gen.num_frames,
#     num_inference_steps=gen.num_inference_steps,
#     guidance_scale=gen.guidance_scale,
# )
# export_to_video(result.frames[0], f"{smoke.id}.mp4", fps=gen.fps)